# ⚡ MY AI STUDIO — BỘ TĂNG TỐC GPU CLOUD (TESLA T4 16GB)
> **Đường truyền cố định Ngrok vĩnh viễn (Chuẩn như Omni Voice — 100% không bao giờ bị Google ngắt kết nối)**
> 
> 1. Vào menu: **Thời gian chạy** ➔ **Thay đổi loại thời gian chạy** ➔ Chọn **T4 GPU** ➔ Lưu.
> 2. Bấm đúng **1 NÚT PLAY (▶️)** duy nhất ở ô bên dưới.
> 3. Đợi ~1 phút, Colab sẽ kết nối vào tên miền cố định: `https://upturned-evict-geologic.ngrok-free.dev`.
> 4. Quay lại **My AI Studio (Tab Dựng Video ➔ Hoán Đổi Mặt)** bấm **'Kiểm Tra Kết Nối'** là thông ngay lập tức!

In [ ]:
#@title ▶️ KHỞI CHẠY BỘ TĂNG TỐC GPU CLOUD (NGROK CỐ ĐỊNH 1 NÚT BẤM)
#@markdown Tự động kết nối đường truyền Ngrok cố định vĩnh viễn không bị Google Colab chặn.

NGROK_AUTHTOKEN = "3JYfiFRrXHoFYeMq5hh219lbnyR_2VF5LDb2AJDpiSo1SefAm" #@param {type:"string"}
NGROK_DOMAIN = "upturned-evict-geologic.ngrok-free.dev" #@param {type:"string"}

import os
import sys
import time
import re
import subprocess

print("=" * 70)
print("⚡ MY AI STUDIO — GPU COMPUTE ACCELERATOR (TESLA T4 16GB)")
print("=" * 70)

# 1. Kiểm tra GPU Tesla T4
try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        print(f"✅ Đã kích hoạt GPU: {gpu_name} (Sẵn sàng xử lý!)")
    else:
        print("⚠️ Lưu ý: Chưa bật GPU! Hãy vào Thời gian chạy ➔ Thay đổi loại thời gian chạy ➔ Chọn T4 GPU!")
except Exception:
    pass

# 2. Cài đặt thư viện môi trường cần thiết
print("\n📦 [1/4] Cài đặt các gói tính toán CUDA song song...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn", "python-multipart", "opencv-python-headless", "insightface", "onnx", "tqdm", "pyngrok"], check=True)

# Kích hoạt onnxruntime-gpu tương thích CUDA 12
try:
    import onnxruntime as ort
    has_cuda = 'CUDAExecutionProvider' in ort.get_available_providers()
except Exception:
    has_cuda = False

if not has_cuda:
    print("  ⏳ Tối ưu bộ đệm CUDA 12 cho GPU...")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "onnxruntime", "onnxruntime-gpu"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnxruntime-gpu", "--extra-index-url", "https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nvidia-cublas-cu12", "nvidia-cudnn-cu12"], check=False)

# 3. Tải mã nguồn Worker
print("\n💾 [2/4] Chuẩn bị động cơ xử lý...")
worker_url = "https://raw.githubusercontent.com/nviethiep55-glitch/my-ai-studio-colab/main/worker.py"
subprocess.run(["wget", "-q", "-O", "worker.py", worker_url], check=False)
if not os.path.exists("worker.py") or os.path.getsize("worker.py") < 100:
    subprocess.run(["curl", "-sL", worker_url, "-o", "worker.py"], check=False)

# 4. Khởi chạy GPU Server
print("\n⚡ [3/4] Khởi động GPU Server...")
subprocess.run(["pkill", "-f", "worker.py"], check=False)
subprocess.run(["pkill", "-f", "ngrok"], check=False)
time.sleep(1)

server_proc = subprocess.Popen([sys.executable, "worker.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
time.sleep(2)

# 5. Kết nối đường truyền Ngrok cố định vĩnh viễn (Chuẩn như Omni Voice)
print("\n🔗 [4/4] Mở đường truyền Ngrok cố định...")
share_url = None
try:
    from pyngrok import ngrok
    ngrok.kill()
    ngrok.set_auth_token(str(NGROK_AUTHTOKEN).strip())
    t_opts = {"addr": 8000, "proto": "http"}
    if NGROK_DOMAIN and str(NGROK_DOMAIN).strip():
        t_opts["domain"] = str(NGROK_DOMAIN).strip()
    tunnel = ngrok.connect(**t_opts)
    share_url = tunnel.public_url.replace("http://", "https://")
    print("\n" + "=" * 70)
    print("🎉 ĐÃ KẾT NỐI ĐƯỜNG TRUYỀN NGROK CỐ ĐỊNH VĨNH VIỄN:")
    print(f"\n👉  {share_url}  👈\n")
    print("💡 Đường dẫn này đã được lưu cố định trong My AI Studio!")
    print("   Bạn chỉ cần bấm 'Kiểm Tra Kết Nối' trên Studio là chạy ngay!")
    print("=" * 70 + "\n")
except Exception as ngrok_err:
    print(f"⚠️ Ngrok gặp lỗi: {ngrok_err}. Đang mở cổng dự phòng Cloudflare...")
    try:
        cf_bin = "/usr/local/bin/cloudflared"
        if not os.path.exists(cf_bin):
            subprocess.run(["wget", "-q", "-nc", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"], check=False)
            subprocess.run(["dpkg", "-i", "cloudflared-linux-amd64.deb"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
        tunnel_log = "/content/tunnel.log"
        subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--logfile", tunnel_log], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        for _ in range(15):
            time.sleep(1)
            if os.path.exists(tunnel_log):
                with open(tunnel_log, "r", errors="ignore") as f:
                    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", f.read())
                    if match:
                        share_url = match.group(0)
                        print(f"🎉 ĐÃ MỞ CLOUDFLARE DỰ PHÒNG: {share_url}")
                        break
    except Exception as cf_e:
        print(f"⚠️ Cloudflare dự phòng lỗi: {cf_e}")

try:
    while True:
        line = server_proc.stdout.readline()
        if line:
            print(line, end="", flush=True)
        time.sleep(0.1)
except KeyboardInterrupt:
    print("\n🛑 Đã dừng GPU Worker!")
    server_proc.terminate()
